<a href="https://colab.research.google.com/github/kimmy111-zhu/human-validation/blob/main/notebooks/alpacafarm_esl_matched_stratified_esl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import json
import random
import os
import pandas as pd
from google.colab import files


# =========================================================
# 1. Settings
# =========================================================

RANDOM_SEED = 42

SAMPLE_SIZE = 5

BENCHMARK_NAME = "AlpacaFarm"


# =========================================================
# 2. Language files
# =========================================================

FILE_PATHS = {
    "Arabic": "/content/arabic_esl_paper.jsonl",
    "French": "/content/french_esl_paper.jsonl",
    "German": "/content/german_esl_paper.jsonl",
    "Japanese": "/content/japanese_esl_paper.jsonl",
    "Mandarin": "/content/mandarin_esl_paper.jsonl",
    "Portuguese": "/content/portuguese_esl_paper.jsonl",
    "Russian": "/content/russian_esl_paper.jsonl",
    "Spanish": "/content/spanish_esl_paper.jsonl"
}


LANGUAGES = list(FILE_PATHS.keys())


# =========================================================
# 3. Check input files
# =========================================================

print("Checking input files:\n")

for language, file_path in FILE_PATHS.items():

    if not os.path.exists(file_path):
        raise FileNotFoundError(
            f"File not found: {file_path}\n"
            f"Please upload the {language} ESL file to Colab."
        )

    print(f"{language}: {file_path}")


# =========================================================
# 4. Helper functions
# =========================================================

def read_jsonl(file_path):
    """
    Read a JSONL file and preserve the original row number.
    """

    records = []

    with open(file_path, "r", encoding="utf-8-sig") as file:

        for line_number, line in enumerate(file, start=1):

            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)

            except json.JSONDecodeError as error:
                raise ValueError(
                    f"Invalid JSON on line {line_number}\n"
                    f"File: {file_path}\n"
                    f"Error: {error}"
                )

            record["_source_row"] = line_number
            records.append(record)

    return records


def normalize_text(value):
    """
    Normalize spaces and line breaks only for matching.

    The original text in the final CSV will not be changed.
    """

    if value is None:
        return ""

    return " ".join(str(value).split())


def convert_to_cell(value):
    """
    Convert lists and dictionaries into readable CSV text.
    """

    if value is None:
        return ""

    if isinstance(value, (list, dict)):
        return json.dumps(
            value,
            ensure_ascii=False
        )

    return str(value)


# =========================================================
# 5. Read all language files
# =========================================================

datasets = {}

print("\nLoading files:\n")

for language, file_path in FILE_PATHS.items():

    datasets[language] = read_jsonl(file_path)

    print(
        f"Loaded {len(datasets[language])} records "
        f"for {language}"
    )


# Show fields in the first record
print("\nFields found in the first record of each file:")

for language in LANGUAGES:

    if datasets[language]:

        print(
            f"\n{language}: "
            f"{list(datasets[language][0].keys())}"
        )


# =========================================================
# 6. Build matching maps using 'instruction'
# =========================================================

record_maps = {}

print("\nBuilding matching maps:")

for language, records in datasets.items():

    current_map = {}

    missing_original_count = 0
    duplicate_count = 0

    for record in records:

        # Original prompt
        original_prompt = record.get(
            "instruction",
            ""
        )

        normalized_original = normalize_text(
            original_prompt
        )

        if not normalized_original:
            missing_original_count += 1
            continue

        if normalized_original in current_map:
            duplicate_count += 1
            continue

        current_map[normalized_original] = record

    record_maps[language] = current_map

    print(
        f"\n{language}: "
        f"{len(current_map)} usable original prompts"
    )

    if missing_original_count > 0:
        print(
            f"Warning: {missing_original_count} records "
            f"had no instruction field."
        )

    if duplicate_count > 0:
        print(
            f"Warning: {duplicate_count} duplicate original prompts "
            f"were found. The first record was kept."
        )


# =========================================================
# 7. Find prompts shared by all 8 languages
# =========================================================

first_language = LANGUAGES[0]

common_original_keys = set(
    record_maps[first_language].keys()
)


for language in LANGUAGES[1:]:

    common_original_keys &= set(
        record_maps[language].keys()
    )


# Sort before sampling for complete reproducibility
common_original_keys = sorted(
    common_original_keys
)


print(
    f"\nCommon original prompts across all "
    f"{len(LANGUAGES)} languages: "
    f"{len(common_original_keys)}"
)


if len(common_original_keys) < SAMPLE_SIZE:

    raise ValueError(
        f"Only {len(common_original_keys)} common prompts "
        f"were found across all language files.\n"
        f"Cannot sample {SAMPLE_SIZE} prompts."
    )


# =========================================================
# 8. Randomly sample 5 common original prompts
# =========================================================

random.seed(RANDOM_SEED)

selected_original_keys = random.sample(
    common_original_keys,
    SAMPLE_SIZE
)


print(f"\nRandom seed: {RANDOM_SEED}")

print(
    f"Selected common original prompts: "
    f"{len(selected_original_keys)}"
)


# =========================================================
# 9. Create matched stratified output
# =========================================================

output_rows = []


for prompt_number, original_key in enumerate(
    selected_original_keys,
    start=1
):

    base_id = (
        f"{BENCHMARK_NAME}_{prompt_number:03d}"
    )

    # Take the original prompt from the first language file
    reference_record = record_maps[first_language][
        original_key
    ]

    original_prompt = reference_record.get(
        "instruction",
        ""
    )

    for language in LANGUAGES:

        record = record_maps[language][
            original_key
        ]

        modified_prompt = record.get(
            "text_transformed",
            ""
        )

        gold_answer = record.get(
            "output",
            ""
        )

        applied_rules = record.get(
            "applied_rules",
            []
        )

        input_text = record.get(
            "input",
            ""
        )

        output_rows.append({

            "Base_ID": base_id,

            "Sample_ID": (
                f"{base_id}_{language}"
            ),

            "Benchmark": BENCHMARK_NAME,

            "Language": language,

            "Source_File": os.path.basename(
                FILE_PATHS[language]
            ),

            "Source_Row": record.get(
                "_source_row",
                ""
            ),

            "Original_Prompt": convert_to_cell(
                original_prompt
            ),

            "Input": convert_to_cell(
                input_text
            ),

            "Modified_Prompt": convert_to_cell(
                modified_prompt
            ),

            "Gold_Answer": convert_to_cell(
                gold_answer
            ),

            "Applied_Rules": convert_to_cell(
                applied_rules
            ),

            # Meaning preservation
            "R1_Meaning": "",
            "R2_Meaning": "",
            "Final_Meaning": "",

            # Key information preservation
            "R1_Key_Info": "",
            "R2_Key_Info": "",
            "Final_Key_Info": "",

            # Realism
            "R1_Realism": "",
            "R2_Realism": "",
            "Final_Realism": "",

            # Readability
            "R1_Readability": "",
            "R2_Readability": "",
            "Final_Readability": "",

            # General comments
            "Comments": ""
        })


# =========================================================
# 10. Convert to DataFrame
# =========================================================

sample_df = pd.DataFrame(
    output_rows
)


print("\nSampling completed.")

print(
    f"Selected original prompts: "
    f"{SAMPLE_SIZE}"
)

print(
    f"Number of language strata: "
    f"{len(LANGUAGES)}"
)

print(
    f"Total output rows: "
    f"{len(sample_df)}"
)


display(sample_df)


# =========================================================
# 11. Check important columns
# =========================================================

empty_original = (
    sample_df["Original_Prompt"]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

empty_modified = (
    sample_df["Modified_Prompt"]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

empty_gold_answer = (
    sample_df["Gold_Answer"]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)


print("\nColumn check:")

print(
    f"Empty Original_Prompt rows: "
    f"{empty_original}"
)

print(
    f"Empty Modified_Prompt rows: "
    f"{empty_modified}"
)

print(
    f"Empty Gold_Answer rows: "
    f"{empty_gold_answer}"
)


# =========================================================
# 12. Check language stratification
# =========================================================

print("\nRows per language stratum:")

language_counts = (
    sample_df["Language"]
    .value_counts()
    .reindex(LANGUAGES)
)

print(language_counts)


# =========================================================
# 13. Check matched sampling
# =========================================================

# Each Base_ID should appear exactly 8 times
base_id_counts = (
    sample_df
    .groupby("Base_ID")
    .size()
)

incorrect_base_ids = base_id_counts[
    base_id_counts != len(LANGUAGES)
]


if len(incorrect_base_ids) == 0:

    print(
        "\nMatched sampling check passed: "
        "every Base_ID has all 8 language versions."
    )

else:

    print(
        "\nWarning: Some Base_ID values do not have "
        "exactly 8 language versions:"
    )

    print(incorrect_base_ids)


# Check that each Base_ID contains all expected languages
expected_languages = set(LANGUAGES)

language_set_check = (
    sample_df
    .groupby("Base_ID")["Language"]
    .apply(lambda values: set(values))
)

incorrect_language_sets = language_set_check[
    language_set_check.apply(
        lambda values: values != expected_languages
    )
]


if len(incorrect_language_sets) == 0:

    print(
        "Language check passed: every Base_ID contains "
        "Arabic, French, German, Japanese, Mandarin, "
        "Portuguese, Russian, and Spanish."
    )

else:

    print(
        "\nWarning: Some Base_ID values are missing "
        "one or more language versions:"
    )

    print(incorrect_language_sets)


# =========================================================
# 14. Check that each language has exactly 5 rows
# =========================================================

incorrect_language_counts = language_counts[
    language_counts != SAMPLE_SIZE
]


if len(incorrect_language_counts) == 0:

    print(
        f"Stratification check passed: every language "
        f"has exactly {SAMPLE_SIZE} rows."
    )

else:

    print(
        "\nWarning: Some language strata do not have "
        f"exactly {SAMPLE_SIZE} rows:"
    )

    print(incorrect_language_counts)


# =========================================================
# 15. Save and download CSV
# =========================================================

output_file = (
    "/content/"
    "AlpacaFarm_ESL_matched_stratified_"
    "sample_n5_per_language_seed42.csv"
)


sample_df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)


print(
    f"\nCSV saved successfully: "
    f"{output_file}"
)


files.download(output_file)

Checking input files:

Arabic: /content/arabic_esl_paper.jsonl
French: /content/french_esl_paper.jsonl
German: /content/german_esl_paper.jsonl
Japanese: /content/japanese_esl_paper.jsonl
Mandarin: /content/mandarin_esl_paper.jsonl
Portuguese: /content/portuguese_esl_paper.jsonl
Russian: /content/russian_esl_paper.jsonl
Spanish: /content/spanish_esl_paper.jsonl

Loading files:

Loaded 805 records for Arabic
Loaded 805 records for French
Loaded 805 records for German
Loaded 805 records for Japanese
Loaded 805 records for Mandarin
Loaded 805 records for Portuguese
Loaded 805 records for Russian
Loaded 805 records for Spanish

Fields found in the first record of each file:

Arabic: ['datasplit', 'dataset', 'instruction', 'input', 'output', 'generator', 'sample_mode', 'text_transformed', 'applied_rules', '_source_row']

French: ['datasplit', 'dataset', 'instruction', 'input', 'output', 'generator', 'sample_mode', 'text_transformed', 'applied_rules', '_source_row']

German: ['datasplit', 'da

,Base_ID,Sample_ID,Benchmark,Language,Source_File,Source_Row,Original_Prompt,Input,Modified_Prompt,Gold_Answer,...,R1_Key_Info,R2_Key_Info,Final_Key_Info,R1_Realism,R2_Realism,Final_Realism,R1_Readability,R2_Readability,Final_Readability,Comments
0,AlpacaFarm_001,AlpacaFarm_001_Arabic,AlpacaFarm,Arabic,arabic_esl_paper.jsonl,285,Write a detailed patent writing for an innovat...,,Write a detailed patent writing for an innovat...,This patent writing details an innovative and ...,...,,,,,,,,,,
1,AlpacaFarm_001,AlpacaFarm_001_French,AlpacaFarm,French,french_esl_paper.jsonl,285,Write a detailed patent writing for an innovat...,,Write a detailed patent writing for an innovat...,This patent writing details an innovative and ...,...,,,,,,,,,,
2,AlpacaFarm_001,AlpacaFarm_001_German,AlpacaFarm,German,german_esl_paper.jsonl,285,Write a detailed patent writing for an innovat...,,We were going to write a detailed patent for a...,This patent writing details an innovative and ...,...,,,,,,,,,,
3,AlpacaFarm_001,AlpacaFarm_001_Japanese,AlpacaFarm,Japanese,japanese_esl_paper.jsonl,285,Write a detailed patent writing for an innovat...,,You could write a detailed patent writing for ...,This patent writing details an innovative and ...,...,,,,,,,,,,
4,AlpacaFarm_001,AlpacaFarm_001_Mandarin,AlpacaFarm,Mandarin,mandarin_esl_paper.jsonl,285,Write a detailed patent writing for an innovat...,,You have to write a detailed patent writing fo...,This patent writing details an innovative and ...,...,,,,,,,,,,
5,AlpacaFarm_001,AlpacaFarm_001_Portuguese,AlpacaFarm,Portuguese,portuguese_esl_paper.jsonl,285,Write a detailed patent writing for an innovat...,,"Using verifiable credentials, digital wallet o...",This patent writing details an innovative and ...,...,,,,,,,,,,
6,AlpacaFarm_001,AlpacaFarm_001_Russian,AlpacaFarm,Russian,russian_esl_paper.jsonl,285,Write a detailed patent writing for an innovat...,,A detailed patent writing for an innovative an...,This patent writing details an innovative and ...,...,,,,,,,,,,
7,AlpacaFarm_001,AlpacaFarm_001_Spanish,AlpacaFarm,Spanish,spanish_esl_paper.jsonl,285,Write a detailed patent writing for an innovat...,,Write a detailed patent for issuing community ...,This patent writing details an innovative and ...,...,,,,,,,,,,
8,AlpacaFarm_002,AlpacaFarm_002_Arabic,AlpacaFarm,Arabic,arabic_esl_paper.jsonl,566,Design a programming problem related to the su...,Dynamic Programming,Design programming problem relate subject give...,Design a programming problem related to Dynami...,...,,,,,,,,,,
9,AlpacaFarm_002,AlpacaFarm_002_French,AlpacaFarm,French,french_esl_paper.jsonl,566,Design a programming problem related to the su...,Dynamic Programming,Design a programming problem looking to the su...,Design a programming problem related to Dynami...,...,,,,,,,,,,



Column check:
Empty Original_Prompt rows: 0
Empty Modified_Prompt rows: 0
Empty Gold_Answer rows: 0

Rows per language stratum:
Language
Arabic        5
French        5
German        5
Japanese      5
Mandarin      5
Portuguese    5
Russian       5
Spanish       5
Name: count, dtype: int64

Matched sampling check passed: every Base_ID has all 8 language versions.
Language check passed: every Base_ID contains Arabic, French, German, Japanese, Mandarin, Portuguese, Russian, and Spanish.
Stratification check passed: every language has exactly 5 rows.

CSV saved successfully: /content/AlpacaFarm_ESL_matched_stratified_sample_n5_per_language_seed42.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>